Q1
here we created a adjacency matrix and a feature matrix, then we pass the info by multiplying (note here adjacency itself has the diagonal as one to contain the feature of self node as well). we multiply it with weight and yes we have succesfully passed the message. we used only one layer and so the messaged passed is only between the immediate neighbour.

In [ ]:
import torch
import torch.nn as nn


class SimpleMessagePassingLayer(nn.Module):

    def __init__(self, in_features, out_features):
        super(SimpleMessagePassingLayer, self).__init__()
        # Learnable weight matrix to update features after aggregation
        self.weight = nn.Parameter(torch.FloatTensor(in_features, out_features))
        self.bias = nn.Parameter(torch.FloatTensor(out_features))

        # Initialize weights cleanly
        nn.init.xavier_uniform_(self.weight)
        nn.init.zeros_(self.bias)

    def forward(self, x, adj):
        """Forward pass of the message passing layer.

        Args:
            x (Tensor): Node features matrix of shape (num_nodes, in_features)
            adj (Tensor): Adjacency matrix of shape (num_nodes, num_nodes)
        """
        # Step 1 & 2: Message & Aggregate via Matrix Multiplication
        # This sums up neighbor features for every node simultaneously.
        aggregated_messages = torch.mm(adj, x)

        # Step 3: Update state by applying learnable weights and bias
        out = torch.mm(aggregated_messages, self.weight) + self.bias

        # Apply a non-linear activation function (ReLU)
        return torch.relu(out)


# --- Verification & Example Usage ---
if __name__ == "__main__":
    # Let's create a dummy graph with 3 nodes:
    # Connection layout: 0 <-> 1 <-> 2  (Node 1 connects to both 0 and 2)
    # Self-loops (diagonal = 1) are added so nodes also include their own features.
    adj_matrix = torch.tensor(
        [[1.0, 1.0, 0.0], [1.0, 1.0, 1.0], [0.0, 1.0, 1.0]], dtype=torch.float32
    )

    # Let's give each node 2 initial features
    # Shape: (3 nodes, 2 features)
    node_features = torch.tensor(
        [[1.0, 10.0], [2.0, 20.0], [3.0, 30.0]], dtype=torch.float32
    )

    print("--- Initial Inputs ---")
    print("Adjacency Matrix:\n", adj_matrix)
    print("\nNode Features:\n", node_features)

    # Instantiate our layer transforming 2 input features into 4 hidden features
    gnn_layer = SimpleMessagePassingLayer(in_features=2, out_features=4)

    # Pass the graph through the layer
    updated_features = gnn_layer(node_features, adj_matrix)

    print("\n--- Output ---")
    print("Updated Node Embeddings Shape:", updated_features.shape)
    print("Updated Node Embeddings:\n", updated_features)

Q2
A Knowledge Graph is a structured, directed multi-graph that represents real-world facts as an interconnected network of entities and relational links.

Unlike a standard homogeneous graph (where connections just mean "Node A is next to Node B"), a knowledge graph is explicitly heterogeneous and multi-relational—meaning every edge has a specific type, label, and semantic meaning.
It is shown in teh form of triplet.
Graph could be multi ditrectional.
Traditional search engines look for keyword matches. Knowledge graphs provide semantic context. If you search for "Apple," a knowledge graph looks at the neighboring nodes (e.g., connected to iPhone vs. connected to Orchard) to instantly deduce whether you mean the tech giant or the fruit.
Because links are typed and structured, algorithms can infer brand-new facts that were never explicitly written down.

Q3 )
Oversmoothing effect - Happens when the node features updates so much that they lost their originality. All the nodes start to look same after averaging a lot.
happens when our neural network has a lot of layers.


how to prevent- DropEdge: Randomly remove a small percentage of edges from the graph during training. This disrupts the communication highway and slows down how fast the features blend together.

Skip Connections / Residuals: Add the original input features ($X$) directly back into the deeper hidden layers (like ResNets). This forces the node to remember its original self-identity no matter how many neighbor messages it takes in.

PairNorm: Apply normalization techniques across nodes at each layer to explicitly keep variance high and force node embeddings to stay distinct from one another.

Over squashing - Over-squashing happens when a node tries to pull information from a huge, far-reaching neighborhood, but the hidden embedding size is too small to hold it all.

In a graph, the number of nodes reachable within $K$ hops grows exponentially relative to the number of layers (especially in "small-world" graphs where everything is highly connected). However, the representation capacity of the target node vector only stays constant. Information from distant nodes gets bottlenecked as it travels along fewer compressed paths.

Graph Rewiring: Mechanically modify the graph layout before feeding it to the GNN. For example, adding virtual edges between long-distance nodes or breaking up highly congested bottleneck pathways.

Anisotropic Aggregation (Attention): Use layers like Graph Attention Networks (GATs). Instead of blindly summing all neighborhood vectors equally, the model learns to prioritize which specific paths matter, dropping the noise before it squashes the vector.


 Tension between stacking layers for a larger receptive field versus losing node distinctiveness.


To solve complex relational tasks, you want a large receptive field (more layers) so a node can understand its broader structural context and global position. However, the moment you add those layers to expand its field of view:

You trigger Over-smoothing, which dilutes the node's local identity into a universal average.

You trigger Over-squashing, which creates massive data traffic jams as exponential neighborhood info is compressed too tightly.

Q4
Considering this dataset , if we dont look it as a graph we see individual transaction , but frauds are actually divided into many transactions. if we make graph we can connect the pattern of fraud.
Why we used GCN as a layer because it can look beyond 1 hop. By stacking two GCN layers, the network calculates its own custom embeddings based on 2-hop neighborhoods (tracking the neighbors of your neighbors), exposing long, multi-step laundering chains that manual feature engineering completely misses.

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.datasets import EllipticBitcoinDataset
from torch_geometric.nn import GCNConv

# 1. Load the Dataset
print("Loading Elliptic Bitcoin Dataset (this may take a moment to download)...")
dataset = EllipticBitcoinDataset(root="./data/Elliptic")
data = dataset[0]

# 2. Preprocess Data and Create Custom Masks
# Labels in this dataset: 0 = Licit, 1 = Illicit, 2 = Unknown
# We only want to train and evaluate on known labels (0 and 1).
labels = data.y

# Create a boolean mask identifying exactly which nodes have known labels (0 or 1)
known_mask = labels < 2

# Create Train/Test splits out of the known nodes (80% train, 20% test)
# We shuffle indices of known nodes to distribute them fairly
known_indices = torch.where(known_mask)[0]
shuffled_indices = known_indices[torch.randperm(len(known_indices))]

split_idx = int(0.8 * len(shuffled_indices))
train_indices = shuffled_indices[:split_idx]
test_indices = shuffled_indices[split_idx:]


# 3. Define the GNN Architecture
class FinancialFraudGCN(nn.Module):

    def __init__(self, in_features, hidden_features, out_classes):
        super(FinancialFraudGCN, self).__init__()
        # Layer 1: Aggregates 1-hop neighbor features
        self.conv1 = GCNConv(in_features, hidden_features)
        # Layer 2: Aggregates 2-hop neighborhood history
        self.conv2 = GCNConv(hidden_features, hidden_features)
        # Final Linear Layer to map hidden states to final class outputs
        self.classifier = nn.Linear(hidden_features, out_classes)

    def forward(self, x, edge_index):
        # Step 1: Pass through first GCN layer + Activation
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=0.3, training=self.training)  # Avoid overfitting

        # Step 2: Pass through second GCN layer + Activation
        x = self.conv2(x, edge_index)
        x = F.relu(x)

        # Step 3: Classify into scores for Licit vs Illicit
        return self.classifier(x)


# 4. Initialize Model, Loss Function, and Optimizer
# Input dimension is 165 (native dataset features), hidden dimension chosen is 64, output classes = 2
model = FinancialFraudGCN(
    in_features=data.num_features, hidden_features=64, out_classes=2
)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=1e-4)

# Move model and data to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
data = data.to(device)
train_indices = train_indices.to(device)
test_indices = test_indices.to(device)


# 5. The Training Loop
def train():
    model.train()
    optimizer.zero_grad()

    # Out shape: [num_nodes, 2]
    out = model(data.x, data.edge_index)

    # Calculate loss ONLY on the designated training nodes
    loss = criterion(out[train_indices], data.y[train_indices])
    loss.backward()  # Backpropagation
    optimizer.step()  # Weight updates
    return loss.item()


# 6. Evaluation Function
@torch.no_grad()
def evaluate(indices):
    model.eval()
    out = model(data.x, data.edge_index)

    # Get the index of the highest score (0 or 1)
    predictions = out[indices].argmax(dim=-1)
    targets = data.y[indices]

    # Calculate Accuracy
    correct = (predictions == targets).sum().item()
    accuracy = correct / len(indices)
    return accuracy


# Run the training pipeline for 100 epochs
print("\n--- Starting Training Pipeline ---")
for epoch in range(1, 101):
    loss = train()
    if epoch % 10 == 0 or epoch == 1:
        train_acc = evaluate(train_indices)
        test_acc = evaluate(test_indices)
        print(
            f"Epoch: {epoch:03d} | Loss: {loss:.4f} | Train Acc: {train_acc:.4f} | Test Acc: {test_acc:.4f}"
        )

ModuleNotFoundError: No module named 'torch_geometric'

In [ ]:
Q6

In [2]:
import numpy as np
import networkx as nx


G = nx.karate_club_graph()
print(
    f"Loaded Karate Club Graph with {G.number_of_nodes()} nodes and {G.number_of_edges()} edges.\n"
)

adjacency_matrix = nx.to_numpy_array(G)


np.set_printoptions(threshold=np.inf, linewidth=150)

print("--- 1. Full 34x34 Adjacency Matrix ---")
print(adjacency_matrix)
print("\n" + "=" * 50 + "\n")



edge_list = list(G.edges())

print(f"--- 2. Full Edge List ({len(edge_list)} connections) ---")

for idx, edge in enumerate(edge_list, 1):
    print(f"Connection {idx:02d}: Node {edge[0]:>2d} <---> Node {edge[1]:>2d}")

Loaded Karate Club Graph with 34 nodes and 78 edges.

--- 1. Full 34x34 Adjacency Matrix ---
[[0. 4. 5. 3. 3. 3. 3. 2. 2. 0. 2. 3. 1. 3. 0. 0. 0. 2. 0. 2. 0. 2. 0. 0. 0. 0. 0. 0. 0. 0. 0. 2. 0. 0.]
 [4. 0. 6. 3. 0. 0. 0. 4. 0. 0. 0. 0. 0. 5. 0. 0. 0. 1. 0. 2. 0. 2. 0. 0. 0. 0. 0. 0. 0. 0. 2. 0. 0. 0.]
 [5. 6. 0. 3. 0. 0. 0. 4. 5. 1. 0. 0. 0. 3. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 2. 2. 0. 0. 0. 2. 0.]
 [3. 3. 3. 0. 0. 0. 0. 3. 0. 0. 0. 0. 3. 3. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [3. 0. 0. 0. 0. 0. 2. 0. 0. 0. 3. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [3. 0. 0. 0. 0. 0. 5. 0. 0. 0. 3. 0. 0. 0. 0. 0. 3. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [3. 0. 0. 0. 2. 5. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 3. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [2. 4. 4. 3. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [2. 0. 5. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.

Q7

In [ ]:

class MyGraph:
    def __init__(self):
        self.nodes = {}
        self.edges = []
        self.edge_attrs = {}

    def add_node(self, node_id, features=None):
        self.nodes[node_id] = features

    def add_edge(self, src, dst, attr=None):
        self.edges.append((src, dst))
        if attr:
            self.edge_attrs[(src, dst)] = attr

    def __repr__(self):
        return f"MyGraph | Nodes: {len(self.nodes)}, Edges: {len(self.edges)}"



g = MyGraph()
g.add_node(0, features=[1.0, 0.5])
g.add_node(1, features=[0.3, 0.8])
g.add_edge(0, 1, attr={'weight': 0.9})

print(g)
print("Nodes:", g.nodes)
print("Edges:", g.edges)